In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
from rank_bm25 import BM25Okapi
import duckdb
import hashlib


In [34]:
passages = load_dataset("unicamp-dl/quati", "quati_1M_passages", trust_remote_code=True)
qrels = load_dataset("unicamp-dl/quati", "quati_1M_qrels", trust_remote_code=True)
queries = load_dataset("unicamp-dl/quati", "quati_all_topics", trust_remote_code=True)


In [39]:
def hash_passage(passages):
    hashs = [hashlib.sha256(passage.encode('utf-8')).hexdigest() for passage in passages['passage']]
    return {'passage_hash': hashs}

def split_passage(passages):
    split = [passage.split() for passage in passages['passage']]
    return {'split': split}

passages_hashed = passages["quati_1M_passages"].map(hash_passage, batched=True, batch_size=1000, num_proc=6)
passages_hashed = passages_hashed.map(split_passage, batched=True, batch_size=1000, num_proc=6)
passages_hashed


Map (num_proc=6):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/1000000 [00:00<?, ? examples/s]

Dataset({
    features: ['passage_id', 'passage', 'passage_hash', 'split'],
    num_rows: 1000000
})

In [40]:
passages_hashed.to_parquet("../data/quati/passages.parquet")
qrels["quati_1M_qrels"].to_parquet("../data/quati/rel_queries_passages.parquet")
queries["quati_all_topics"].to_parquet("../data/quati/queries.parquet")


Creating parquet from Arrow format:   0%|          | 0/1000 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

11642

In [41]:
p = duckdb.read_parquet("../data/quati/passages.parquet") # Passages



In [42]:
rel = duckdb.read_parquet("../data/quati/rel_queries_passages.parquet") # Relations between queries and passages


In [43]:
q = duckdb.read_parquet("../data/quati/queries.parquet") # Queries

In [44]:
duckdb.sql("""
SELECT queries.query_id, queries.query, passages.passage, qrels.score, passages.passage_hash FROM "../data/quati/queries.parquet" as queries
inner join "../data/quati/rel_queries_passages.parquet" as qrels
    on queries.query_id = qrels.query_id
inner join "../data/quati/passages.parquet" as passages
    on qrels.passage_id = passages.passage_id
where queries.query_id = 95
order by queries.query, qrels.score desc limit 10
""")

┌──────────┬──────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

In [45]:
passages = p.to_df()
passages

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,passage_id,passage,passage_hash,split
0,clueweb22-pt0000-00-00003_1,"Se você precisar de ajuda, visite o website na...",fa017fd75545011f942f89898d71a5cc12b757f445da87...,"[Se, você, precisar, de, ajuda,, visite, o, we..."
1,clueweb22-pt0000-00-00003_2,Qual a parada final de Sintra para Cascais ôni...,d6f25691f2b27a287d9ca16a37b0c3db104fe55e7a5104...,"[Qual, a, parada, final, de, Sintra, para, Cas..."
2,clueweb22-pt0000-00-00003_3,"""O Casino Estoril fica localizado no Estoril, ...",85867396297dae9b25395eb98e98a509bc07d403040c75...,"[""O, Casino, Estoril, fica, localizado, no, Es..."
3,clueweb22-pt0000-00-00003_5,Sintra para Lisboa Sintra para Óbidos Leiria P...,c722ca9dbaab3a0cd004c8a6a91924788f5139bbd7ae04...,"[Sintra, para, Lisboa, Sintra, para, Óbidos, L..."
4,clueweb22-pt0000-00-00008_0,Iniciar a digitalização Iniciar a digitalizaçã...,892ec06ad30be1c5bf6cc824ed95df3283d6b961464c0c...,"[Iniciar, a, digitalização, Iniciar, a, digita..."
...,...,...,...,...
999995,clueweb22-pt0002-45-01752_2,"Pararam totalmente suas vidas, reagiram parand...",a0f7dc054d18fb9e3905c2786b3fb60c358cc275e10313...,"[Pararam, totalmente, suas, vidas,, reagiram, ..."
999996,clueweb22-pt0002-45-01757_0,Recicla Eletro - Coleta Gratuita de Lixo Eletr...,3db90172eb9bde6020b8d7f0bba1a0622e01c5ec2a0a5e...,"[Recicla, Eletro, -, Coleta, Gratuita, de, Lix..."
999997,clueweb22-pt0002-45-01760_4,É importante falar de liberdade ao falar de Ro...,7c27623ca11424ffab463716f4e0dbf72dc684da109757...,"[É, importante, falar, de, liberdade, ao, fala..."
999998,clueweb22-pt0002-45-01762_5,Os artigos 2º e artigos 84 a 86 da Lei nº 13.1...,6a25ca297e75fa2c2bc5cbf49cf429a85202e1306059ee...,"[Os, artigos, 2º, e, artigos, 84, a, 86, da, L..."


In [32]:
passages = p.select('passage,passage_hash')
tokenized_corpus = [doc[0].split(" ") for doc in passages.fetchall()]


In [33]:
passages.fetchone()

('Se você precisar de ajuda, visite o website nacional sobre a COVID-19 ou ligue para a linha de apoio à COVID-19 808 24 24 24 Perguntas mais frequentes Posso viajar entre Sintra e Cascais? Quais são as restrições de viagem em Cascais? Qual o número de telefone de apoio para a COVID 19 em Cascais? Preciso utilizar máscara facial no transporte público em Cascais? A prática do distanciamento social é compulsória em Cascais? O que eu devo fazer caso apresente sintomas da COVID-19 quando chegar em Cascais? Última atualização: 25 Abr 2022 Aplicam-se exceções, para detalhes completos: European Union. Estamos trabalhando ininterruptamente para lhe trazer as últimas informações de viagem relacionadas à COVID-19. Esta informação é compilada a partir de fontes oficiais. Ao melhor de nosso conhecimento, está correta de acordo com a última atualização. Visite Avisos de Viagem Rome2rio para ajuda geral. Perguntas & Respostas Qual a maneira mais econômica de ir de Sintra para Cascais? Qual a maneira

In [21]:
df = pd.DataFrame(quati_1M_qrels['quati_1M_qrels'])
for _, row in df.query("query_id == 1").sort_values("score", ascending=False).head(10).iterrows():
    print(df_query.query(f"passage_id == '{row['passage_id']}'")["passage"].values[0])


A fauna brasileira está entre as mais diversas do mundo. Entretanto, toda essa riqueza vem sendo constantemente ameaçada devido a execrável prática de comércio ilícito de diversas espécies. Desse modo... Redação completa Enviada em: 02/09/2017 Vida Os animais da fauna brasileira são variados, diversificados em cores, formas e espécies, possuem uma pluralidade única, em todo o mundo, e por conseguinte, atraem a atenção daqueles,... Redação completa Enviada em: 07/09/2017 Desde o início do século XVI, com a chegada oficial da colonização portuguesa ao Brasil, cultiva-se a ideia de que a fauna e flora da pátria são infinitas. Nesse sentido o homem começou a apropriar-se... Redação completa Enviada em: 08/09/2017 Tráfico. Roubo. Desenvolvimento. Progresso, são pilares da biopirrataria. Hodiernamente, o número de espécies silvestres contrabandeadas tem uma tendência ao aumento, pois os países desenvolvidos gara... Redação completa Enviada em: 14/09/2017 Relativo o combate ao comércio ilegal

In [7]:
tokenized_corpus = [doc.split(" ") for doc in quati_1M["quati_1M_passages"]["passage"]]
tokenized_corpus

[['Se',
  'você',
  'precisar',
  'de',
  'ajuda,',
  'visite',
  'o',
  'website',
  'nacional',
  'sobre',
  'a',
  'COVID-19',
  'ou',
  'ligue',
  'para',
  'a',
  'linha',
  'de',
  'apoio',
  'à',
  'COVID-19',
  '808',
  '24',
  '24',
  '24',
  'Perguntas',
  'mais',
  'frequentes',
  'Posso',
  'viajar',
  'entre',
  'Sintra',
  'e',
  'Cascais?',
  'Quais',
  'são',
  'as',
  'restrições',
  'de',
  'viagem',
  'em',
  'Cascais?',
  'Qual',
  'o',
  'número',
  'de',
  'telefone',
  'de',
  'apoio',
  'para',
  'a',
  'COVID',
  '19',
  'em',
  'Cascais?',
  'Preciso',
  'utilizar',
  'máscara',
  'facial',
  'no',
  'transporte',
  'público',
  'em',
  'Cascais?',
  'A',
  'prática',
  'do',
  'distanciamento',
  'social',
  'é',
  'compulsória',
  'em',
  'Cascais?',
  'O',
  'que',
  'eu',
  'devo',
  'fazer',
  'caso',
  'apresente',
  'sintomas',
  'da',
  'COVID-19',
  'quando',
  'chegar',
  'em',
  'Cascais?',
  'Última',
  'atualização:',
  '25',
  'Abr',
  '2022',
  

In [8]:
bm25 = BM25Okapi(tokenized_corpus)

In [11]:
query = quati_all_topics['quati_all_topics'][0]['query']
tokenized_query = query.split(" ")
print(quati_all_topics['quati_all_topics'][0])
doc_scores = bm25.get_scores(tokenized_query)
print(quati_all_topics['quati_all_topics'][0:10])
doc_scores.shape

{'query_id': 0, 'query': 'Qual a maior característica da flora brasileira?'}
{'query_id': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'query': ['Qual a maior característica da flora brasileira?', 'Qual a maior característica da fauna brasileira?', 'Por que os países Guiana e Suriname não são filiados a Conmebol?', 'Por que a Bolívia tem duas capitais?', 'Qual o que significa decolonialidade?', 'O que é sistema Swift?', 'Por que a seleção uruguaia ostenta 4 estrelas em seu escudo?', 'Qual a importância da Semana de Arte Moderna de 1922?', 'Por que Johann Sebastian Bach é considerado um revolucionário na história da música?', 'quais países europeus seguem o regime monarquista?']}


(1000000,)

In [12]:
doc_scores

array([16.7759861 , 12.69478302, 14.7678286 , ...,  6.50254305,
       12.34798885, 12.66958091], shape=(1000000,))

In [16]:
for t in bm25.get_top_n(tokenized_query, tokenized_corpus, n=10):
    print(" ".join(t))

Maiores Prêmios da Mega Sena | Intersena LOTERIA DETALHES TOTAL Nenhum item no seu carrinho de compras. Total: R$ 0,00 Maiores Prêmios da Mega Sena já pagos | Guia das Loterias Maiores Prêmios da Mega Sena já pagos | Guia das Loterias Maiores prêmios da mega sena já pagos O que você faria se ficasse milionário? Acertar as 6 dezenas na loteria do sonho de muitos brasileiros e ser ganhador dos Maiores prêmios da Mega-Sena parece impossível. Mas a verdade é que o concurso da Mega-Sena já pagou prêmios milionários e tornou possível a sorte entre 60 números. Quanto mais jogadores apostarem na Mega-Sena, maior será a arrecadação e o valor a ser pago aos vencedores. O valor do prêmio também acumula caso ninguém acerte as 6 dezenas sorteadas. Esses valores em dinheiro com certeza mudaram a vida de muitos apostadores. E você sabe qual foi o maior prêmio que a Mega-Sena já pagou? Quer saber de onde são os ganhadores da loteria brasileira? Então fique ligado que nesse artigo contaremos tudo para 

In [ ]:
query = "windy London"
tokenized_query = query.split(" ")

doc_scores = bm25.get_scores(tokenized_query)

In [26]:
print('Describe:', df_quati['palavras'].describe())

Describe: count    1000000.000000
mean         177.113572
std           36.832993
min           50.000000
25%          162.000000
50%          186.000000
75%          202.000000
max          350.000000
Name: palavras, dtype: float64


In [27]:
print('Describe:', df_quati['tokens'].describe())


Describe: count    1000000.000000
mean         331.813221
std           92.518719
min           59.000000
25%          284.000000
50%          330.000000
75%          377.000000
max         1992.000000
Name: tokens, dtype: float64
